# AlpsML · Modelos UV de ALPs con XGBoost + SHAP + MCMC

## Motivación física

Las **partículas tipo axión (ALPs)** surgen de la ruptura espontánea de una
simetría global U(1)<sub>PQ</sub> a escala f<sub>a</sub>. En un modelo
UV-completo, los acoples del ALP a las partículas del SM **no son parámetros
libres**: están completamente determinados por las **cargas de Peccei–Quinn
(PQ)** asignadas a los fermiones del SM.

La conexión entre el nivel UV y los acoples IR observables sigue dos reglas:

1. **Acoples a fermiones**: son directamente iguales a la carga PQ del
   fermión correspondiente (salvo por un signo procedente de la estructura
   Yukawa).
2. **Acoples a bosones gauge** (gluones, fotones, W/Z): son combinaciones
   lineales de las cargas PQ, fijadas por los **coeficientes de anomalía
   quiral** de la corriente PQ con cada grupo gauge:

$$
g_{aGG} \propto \sum_f T(R_f)\,Q_f^{\rm PQ}, \qquad
g_{a\gamma\gamma} \propto \sum_f Q_f^{\rm em,2}\,Q_f^{\rm PQ}.
$$

Esto es lo que implementa `alpaca` a través de `PQChargedModel.get_couplings`:
dados los 5 parámetros de carga PQ más f<sub>a</sub>, calcula automáticamente
el conjunto completo de acoples IR y evalúa el χ² sobre las transiciones de
mesones.

### Modelo no-universal

Las cargas son **dependientes de generación** para los dobletes de SU(2)<sub>L</sub>:
solo la **tercera generación** de q<sub>L</sub> y l<sub>L</sub> tiene carga
PQ no nula, mientras que los singletes derechos (u<sub>R</sub>, d<sub>R</sub>,
e<sub>R</sub>) son universales en generación:

```python
from alpaca.uvmodels import PQChargedModel

model = PQChargedModel('non-universal model', {
    'qL': [0, 0, pq_qL],   # doblete izquierdo de quarks: carga solo en 3ª gen
    'lL': [0, 0, pq_lL],   # doblete izquierdo de leptones: carga solo en 3ª gen
    'uR': pq_uR,            # singlete derecho up: universal
    'dR': pq_dR,            # singlete derecho down: universal
    'eR': pq_eR,            # singlete derecho cargado: universal
})
couplings = model.get_couplings({}, 4 * np.pi * fa)
```

El espacio de parámetros explorado es, por tanto, de 6 dimensiones UV:
`log_fa`, `pq_qL`, `pq_lL`, `pq_uR`, `pq_dR`, `pq_eR`.

---

## Pipeline

1. **Dataset χ²**: 15 000 puntos en Latin Hypercube 6D sobre los 5 parámetros
   de carga PQ y `log_fa`. Para cada punto se construye el `PQChargedModel`,
   se obtienen los acoples IR vía `get_couplings`, y se evalúa el χ² combinado
   de las 5 transiciones con `alpaca`. Paralelo con `multiprocessing.Pool` y
   backup cada 100 puntos. Target sigmoide centrado en χ² = 10.
2. **XGBoost**: regresor del target ponderado (puntos excluidos pesan ×5).
   Búsqueda Optuna (100 trials) + entrenamiento final con early stopping.
   Profundidad 2–6, regularización α, λ ∈ [0.1, 10].
3. **SHAP**: `Permutation Explainer` sobre el surrogate. Beeswarm, bar,
   waterfall y dependence plots para entender qué cargas PQ controlan
   la fenomenología.
4. **MCMC**: 50 walkers × 5 000 pasos con `emcee` sobre el surrogate;
   `corner` plot del posterior con burn-in 1 000 y thin 15.

### Cachés

Las dos etapas pesadas (generación del dataset y entrenamiento) están **gateadas**:
solo corren si el archivo no existe **o** si pones a `True` el flag correspondiente
en el bloque de Setup (`FORCE_REGENERATE`, `FORCE_RETRAIN`).

## Índice

| §   | Sección                                  | Caché                              | Coste aprox. |
|-----|------------------------------------------|------------------------------------|--------------|
| [Setup](#setup)        | Imports, paths, flags             | —                                  | <1 s |
| [§1](#sec-1)           | Generación del dataset χ²         | `outputs/datasets/dataset_alps_uv.csv` | minutos (15 000 evals × `alpaca`) |
| [§2](#sec-2)           | Inspección del dataset            | —                                  | <1 s |
| [§3](#sec-3)           | Entrenamiento XGBoost (Optuna)    | `outputs/models/modelo_alps_uv.json` | ~5 min (100 trials × 1000 rondas) |
| [§4](#sec-4)           | Interpretabilidad SHAP            | —                                  | ~30 s |
| [§5](#sec-5)           | Posterior MCMC + corner plot      | `outputs/datasets/posterior_samples_uv.csv` | ~1 min (50 walkers × 5 000 pasos) |


## <a id="setup"></a>Setup (ejecutar primero)

Define paths via `pathlib`, crea los directorios de salida si no existen, y fija
los flags `FORCE_REGENERATE` / `FORCE_RETRAIN`. También limita BLAS a 1 thread por
proceso (la generación paralelizará a mano con `multiprocessing.Pool`).


In [ ]:
import os

# BLAS: 1 thread por proceso (la generacion reparte por mp.Pool)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

from pathlib import Path

# --- Paths ---
ROOT = Path.cwd().parent
PAPER_DIR    = Path.home() / "alejandro" / "alpsml"
DATASETS_DIR = PAPER_DIR / "datasets"
MODELS_DIR   = PAPER_DIR / "models"
FIGURES_DIR  = PAPER_DIR / "figures"
for d in (DATASETS_DIR, MODELS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATASET_CSV          = DATASETS_DIR / "dataset_alps_uv_v2.csv"
DATASET_BACKUP       = DATASETS_DIR / "backup_dataset_alps_uv_v2.csv"

# --- DOS surrogates (patron de Jorge-Alda/comcha_tutorial, autores de alpaca) ---
#   * clasificador (CLF): aprende sigmoid(chi2) -> frontera permitido/excluido.
#                         En el MCMC se usa SOLO como muro blando (penalty).
#   * regresor   (REG):  aprende chi2 CRUDO en la region permitida.
#                         Da el likelihood real  log L = -0.5 * chi2.
CLF_PATH        = MODELS_DIR / "modelo_alps_clf_v2.json"
REG_PATH        = MODELS_DIR / "modelo_alps_reg_v2.json"
BEST_PARAMS_CLF = MODELS_DIR / "best_params_clf_v2.json"
BEST_PARAMS_REG = MODELS_DIR / "best_params_reg_v2.json"

POSTERIOR_CSV        = DATASETS_DIR / "posterior_samples_uv_v2.csv"
POSTERIOR_CSV_THEORY = DATASETS_DIR / "posterior_samples_uv_v2_theory.csv"
CORNER_PNG           = FIGURES_DIR  / "corner_plot_uv_v2.png"
CORNER_PHYS_PNG      = FIGURES_DIR  / "corner_plot_uv_v2_phys.png"

# --- Config del surrogate (compartida por dataset + training + MCMC) ---
# El corte define la region permitida como  Delta-chi2 = CUTOFF sobre el minimo
# (mejor ajuste), NO un chi2 absoluto.
CUTOFF        = 10.0
SIGMOID_WIDTH = 0.4

# --- Cores para generacion del dataset ---
# La generacion es CPU-bound (get_chi2 ~1.6 s/punto, single-thread por
# OMP_NUM_THREADS=1), asi que conviene MAXIMIZAR workers -> todos los cores.
N_CORES = 16   # maximo 16 cores en el cluster (bonden); main usa os.cpu_count()

# --- Flags de regeneracion ---
# Por defecto a False: al re-ejecutar se usan las CACHES en disco (dataset,
# modelos y posterior) y NO se repite el calculo pesado. Pon a True solo lo que
# quieras rehacer (p.ej. FORCE_REGENERATE=True tras cambiar el rango de ma).
FORCE_REGENERATE = False   # 1 -- regenera el dataset
FORCE_RETRAIN    = False   # 3 -- reentrena XGBoost (clasificador + regresor)
FORCE_MCMC       = False   # 5 -- re-corre MCMC

# --- Limites fisicos ---
# Cargas PQ en [-1, 1] (confirmado por el profesor).
PQ_MIN = -1.0
PQ_MAX =  1.0
# Ventana de masa del ALP.  << RANGO INFERIDO, confirmar con el profesor >>
# El corner del profesor muestra ma in [1.65, 2.10] GeV (pico ~1.80). Un scan
# libre en [0.2, 4.5] cae en ma~0.3 (minimo global del chi2 en NUESTRA config),
# asi que restringimos a una ventana centrada en ~1.8 para reproducir su region.
MA_MIN = 1.5
MA_MAX = 2.5

FEATURES = ["log_fa", "pq_qL", "pq_lL", "pq_uR", "pq_dR", "pq_eR", "ma"]

print("Setup OK")
print(f"  ROOT       = {ROOT}")
print(f"  cores      = {N_CORES}")
print(f"  ma window  = [{MA_MIN}, {MA_MAX}] GeV   |   PQ in [{PQ_MIN}, {PQ_MAX}]")
print(f"  dataset    = {DATASET_CSV} ({'exists' if DATASET_CSV.exists() else 'missing'})")
print(f"  clf model  = {CLF_PATH} ({'exists' if CLF_PATH.exists() else 'missing'})")
print(f"  reg model  = {REG_PATH} ({'exists' if REG_PATH.exists() else 'missing'})")
print(f"  posterior  = {POSTERIOR_CSV} ({'exists' if POSTERIOR_CSV.exists() else 'missing'})")
print(f"  FORCE_REGENERATE={FORCE_REGENERATE}  FORCE_RETRAIN={FORCE_RETRAIN}  FORCE_MCMC={FORCE_MCMC}")


## <a id="sec-1"></a>§1. Generación del dataset χ²

### Construcción del modelo UV

Para cada punto del espacio de parámetros se instancia un `PQChargedModel`
no-universal. Los 5 parámetros de carga PQ determinan automáticamente todos
los acoples IR del ALP:

- **Acoples a fermiones**: proporcionales a la carga PQ del fermión (con un
  signo que depende de si el campo transforma por la izquierda o por la
  derecha bajo la simetría PQ).
- **Acoples a gluones y fotones**: calculados por `alpaca` a partir de los
  coeficientes de anomalía de la corriente PQ, que son sumas ponderadas de
  las cargas PQ de todos los fermiones coloreados/cargados.

El modelo no-universal específico es:

```
'qL': [0, 0, pq_qL]  →  solo la 3ª generación del doblete izquierdo de quarks tiene carga PQ
'lL': [0, 0, pq_lL]  →  solo la 3ª generación del doblete izquierdo de leptones tiene carga PQ
'uR': pq_uR           →  singlete derecho up: carga universal en las 3 generaciones
'dR': pq_dR           →  singlete derecho down: carga universal en las 3 generaciones
'eR': pq_eR           →  singlete derecho cargado: carga universal en las 3 generaciones
```

La llamada `model.get_couplings({}, 4*np.pi*fa)` devuelve el conjunto completo
de acoples IR (incluyendo los de gluones y fotones derivados de las anomalías),
que `alpaca` usa para evaluar los observables de cada transición.

### Muestreo y χ²

Latin Hypercube en 6 dimensiones sobre
$(\log f_a,\; p_{qL},\; p_{lL},\; p_{uR},\; p_{dR},\; p_{eR})$
con `scipy.stats.qmc.LatinHypercube`. Para cada punto se calcula el χ²
combinado de las 5 transiciones objetivo, que son los procesos con mejores
vínculos experimentales para ALPs de ~2 GeV:

| Transición | Canal |
|------------|-------|
| $K^+ \to a\,\pi^+$ | FCNC quark $s \to d$ mediado por el ALP |
| $K^0_L \to a\,\pi^0$ | Análogo neutro |
| $B^+ \to K^+\,a$ | FCNC quark $b \to s$ |
| $B^0 \to K^0\,a$ | Análogo neutro |
| $B^+ \to a\,\pi^+$ | FCNC quark $b \to u$ |

Los NaN físicos (canales cinemáticamente cerrados para ciertos valores de
los acoples) se tratan como contribución 0 con `np.nansum`.

El **target sigmoide** $1/(1 + e^{(\chi^2 - 10)/0.4})$ convierte el χ²
en una probabilidad suave: ~1 para puntos muy permitidos (χ² ≪ 10) y ~0
para puntos muy excluidos (χ² ≫ 10), con la frontera experimental en χ² ≈ 10.

**Solo se regenera si** `FORCE_REGENERATE = True` **o** si
`dataset_alps_uv.csv` no existe. Backup cada 100 puntos.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import qmc
import alpaca
import alpaca.sectors as _alpaca_sectors
from alpaca.uvmodels import PQChargedModel
from alpaca.statistics import get_chi2
import warnings
from tqdm.auto import tqdm
import multiprocessing as mp

# --- CONFIGURACION FISICA v2 ---
#   1. Masa del ALP como parametro libre dentro de [MA_MIN, MA_MAX] (ver Setup).
#   2. Cargas PQ en [-1, 1].
#   3. LISTA COMPLETA de observables (recomendacion explicita del profesor):
#      hay que incluir B->K mu mu (ALP que decae de forma VISIBLE), meson mixing
#      (delta_mB0/Bs/K0, epsK, ...) y desintegraciones radiativas/leptonicas.
#      El criterio visible/invisible es especifico de cada experimento (tamano del
#      detector + boost): un mismo ALP puede ser invisible en Belle II pero visible
#      en LHCb, y si es visible alli el chi2 sube mucho. Por eso se pasa el SECTOR
#      ENTERO 'all' tal cual (como en alpaca_chi2.py del tutorial), sin filtrar.
#   4. chi2 via  get_chi2(...).chi2_tot()[0]:
#      * chi2_tot() ya EXCLUYE del fit los observables NaN (no definidos en ese
#        punto: p.ej. B->K a(->tau tau) solo acota si ma>2*m_tau; un decay
#        invisible solo si el ALP es de vida larga). NO hacemos np.nansum a mano.
#      * SIN integrator='no_rge': get_couplings da los acoples a escala alta
#        (4*pi*fa) y get_chi2 debe correrlos (RGE). Sin ese running el chi2 sale
#        mal (comprobado: factores x2-x16). El 'no_rge' del tutorial solo vale
#        alli porque pre-corren con match_run.

# --- Numero de puntos ---
# get_chi2 cuesta ~1.6 s/punto (cuello de botella; no se puede batchear ni evitar
# el RGE). Con la ventana ma in [1.5, 2.5] el volumen del espacio cae ~4x respecto
# a [0.2, 4.5], asi que 8000 puntos dan MAS densidad que los 20k originales sobre
# el rango ancho.  Coste: 8000 / 24 cores ~ 9 min.
#   - sube a 12000 (~14 min) si quieres mas robustez del surrogate
#   - baja a 6000  (~7 min) para mas velocidad
N_PUNTOS = 8_000

# Espacio 7D: (log_fa, pq_qL, pq_lL, pq_uR, pq_dR, pq_eR, ma)
L_BOUNDS_GEN = [6.0, PQ_MIN, PQ_MIN, PQ_MIN, PQ_MIN, PQ_MIN, MA_MIN]
U_BOUNDS_GEN = [8.0, PQ_MAX, PQ_MAX, PQ_MAX, PQ_MAX, PQ_MAX, MA_MAX]

# --- Sector COMPLETO de alpaca (se pasa tal cual a get_chi2) ---
try:
    TRANSICIONES_TARGET = _alpaca_sectors.default_sectors['all']
    _n_obs = len(list(TRANSICIONES_TARGET.observables))
    print(f"Usando el sector 'all' COMPLETO: {_n_obs} observables")
except Exception as _exc:
    warnings.warn(f"No se pudo cargar alpaca.sectors: {_exc}. Usando lista minima.")
    # Fallback con los nombres REALES de alpaca ('alp', 'pion+', no 'a'/'pi+')
    TRANSICIONES_TARGET = [
        "K+ -> alp pion+", "KL -> alp pion0",
        "B+ -> K+ alp", "B0 -> K0 alp", "B+ -> alp pion+",
    ]

# 'fork' imprescindible en Python 3.12+ y notebooks (workers ven funciones de __main__)
try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass
_MP_CTX = mp.get_context('fork')

def obtener_chi2_uv(p):
    """Chi2 global UV para un punto 7D (log_fa, pq_qL, pq_lL, pq_uR, pq_dR, pq_eR, ma)."""
    logfa, pq_qL, pq_lL, pq_uR, pq_dR, pq_eR, ma = p
    fa = 10**logfa
    fa_scale = 4 * np.pi * fa
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        uv_model = PQChargedModel('non-universal model', {
            'qL': [0, 0, pq_qL],
            'lL': [0, 0, pq_lL],
            'uR': pq_uR,
            'dR': pq_dR,
            'eR': pq_eR,
        })
        c = uv_model.get_couplings({}, fa_scale)
        chi2 = get_chi2(TRANSICIONES_TARGET, ma, c, fa)   # RGE por defecto; NaN -> excluidos
        return float(chi2.chi2_tot()[0])

def procesar_punto_paralelo(args):
    i, p = args
    try:
        chi_val = obtener_chi2_uv(p)
        return {
            'log_fa': p[0], 'pq_qL': p[1], 'pq_lL': p[2],
            'pq_uR': p[3], 'pq_dR': p[4], 'pq_eR': p[5], 'ma': p[6],
            'chi2': chi_val,
        }
    except Exception as e:
        return {"error": f"Punto {i}: {e}"}

# --- BLOQUE PRINCIPAL ---
if not FORCE_REGENERATE and DATASET_CSV.exists():
    print(f"Dataset ya existe en {DATASET_CSV} ({DATASET_CSV.stat().st_size/1e6:.1f} MB) -- usando cache.")
    print("Para regenerar, pon FORCE_REGENERATE=True en la celda de Setup.")
elif __name__ == '__main__':
    sampler_lhs = qmc.LatinHypercube(d=7)
    puntos = qmc.scale(sampler_lhs.random(n=N_PUNTOS), L_BOUNDS_GEN, U_BOUNDS_GEN)
    tareas = list(enumerate(puntos))
    print(f"Iniciando calculo paralelo | Puntos: {N_PUNTOS} | Nucleos: {N_CORES} | ~{N_PUNTOS*1.6/N_CORES/60:.0f} min")
    dataset = []
    n_errors = 0
    # maxtasksperchild alto: con ~333 tareas/worker no recicla (evita el coste de
    # re-fork); sigue siendo red de seguridad para runs gigantes.
    with _MP_CTX.Pool(processes=N_CORES, maxtasksperchild=1000) as pool:
        try:
            for resultado in tqdm(
                pool.imap_unordered(procesar_punto_paralelo, tareas, chunksize=2),
                total=N_PUNTOS, desc="Progreso"
            ):
                if resultado is not None:
                    if "error" in resultado:
                        n_errors += 1
                        if n_errors <= 5 or n_errors % 500 == 0:
                            print(f"\n[!] {resultado['error']}")
                    else:
                        dataset.append(resultado)
                if len(dataset) > 0 and len(dataset) % 500 == 0:
                    pd.DataFrame(dataset).to_csv(DATASET_BACKUP, index=False)
                    print(f"  Backup: {len(dataset)} puntos validos ({n_errors} errores)")
        except Exception as pool_err:
            print(f"\n[Pool error] {pool_err} — guardando lo que hay...")
    print(f"\nLoop terminado: {len(dataset)} puntos validos, {n_errors} errores")
    if dataset:
        df = pd.DataFrame(dataset)
        # Descartar chi2 no finitos (puntos patologicos) antes de calibrar el corte
        n_before = len(df)
        df = df[np.isfinite(df['chi2'])].reset_index(drop=True)
        if len(df) < n_before:
            print(f"  Descartados {n_before - len(df)} puntos con chi2 no finito")
        # --- Target sigmoide RELATIVO al minimo (Delta-chi2), como el tutorial ---
        chi2_min = float(df['chi2'].min())
        df['target'] = 1 / (1 + np.exp((df['chi2'] - chi2_min - CUTOFF) / SIGMOID_WIDTH))
        df.to_csv(DATASET_CSV, index=False)
        print(f"chi2_min = {chi2_min:.2f}  ->  region permitida: chi2 < {chi2_min + CUTOFF:.2f}")
        print(f"Guardado: {DATASET_CSV}  ({len(df)} filas)")


## <a id="sec-2"></a>§2. Inspección del dataset

Estadística descriptiva del χ² y conteo de puntos según la target sigmoide:

- **Permitidos** (target > 0.8): zona allowed por las transiciones K/B.
- **Excluidos** (target < 0.2): zona excluida.
- **Frontera** (0.2 ≤ target ≤ 0.8): puntos sobre la curva de transición — los más informativos para el surrogate.


In [ ]:
import pandas as pd

df = pd.read_csv(DATASET_CSV)
print("--- Estad\u00edsticas del Chi2 ---")
print(df['chi2'].describe())

permitidos = len(df[df['target'] > 0.8])
excluidos  = len(df[df['target'] < 0.2])
frontera   = len(df[(df['target'] >= 0.2) & (df['target'] <= 0.8)])

print(f"\nPermitidos (Target ~1): {permitidos}")
print(f"Excluidos  (Target ~0): {excluidos}")
print(f"En la frontera:        {frontera}")


## <a id="sec-3"></a>§3. Entrenamiento XGBoost — dos surrogates

Siguiendo el tutorial oficial de `alpaca`
([Jorge-Alda/comcha_tutorial](https://github.com/Jorge-Alda/comcha_tutorial)),
se entrenan **dos** modelos con roles distintos:

| Modelo | Target | Datos | Uso en el MCMC |
|--------|--------|-------|----------------|
| **CLF** (clasificador) | $\mathrm{sigmoid}(\chi^2)$ | todo el espacio | **muro**: penaliza la zona excluida |
| **REG** (regresor) | $\chi^2$ **crudo** | solo $\chi^2 < \chi^2_{\min}+10$ | **likelihood**: $\log L = -\tfrac12\,\chi^2$ |

La clave (corrección sobre la versión anterior): el **likelihood** sale del
**regresor de $\chi^2$ crudo**, no del sigmoide. El sigmoide solo define la
frontera permitido/excluido y se usa como pared para que los walkers no entren
donde el regresor extrapola. Muestrear `log(sigmoide)` directamente — como se
hacía antes — da una meseta plana y un corner plot "a cajas".

Cada modelo se ajusta con Optuna (árboles poco profundos, regularización fuerte)
y se cachea en `outputs/models/`. **Solo reentrena si** `FORCE_RETRAIN = True` **o**
si faltan los `.json`.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import sklearn.metrics
import json
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# ============================================================================
# DOS surrogates (patron del tutorial de alpaca):
#   CLF  -> regresa sigmoid(chi2) en TODO el espacio (frontera permitido/excluido)
#   REG  -> regresa chi2 CRUDO, entrenado SOLO en la region permitida
# El MCMC usara  log L = -0.5*REG(x)  como likelihood y  CLF(x)  como muro.
# ============================================================================
features = FEATURES  # 7D

df = pd.read_csv(DATASET_CSV)
df = df[np.isfinite(df['chi2'])].reset_index(drop=True)
chi2_min = float(df['chi2'].min())

# Recalcular el target sigmoide RELATIVO al minimo (robusto frente a CSV viejos)
df['target'] = 1.0 / (1.0 + np.exp((df['chi2'] - chi2_min - CUTOFF) / SIGMOID_WIDTH))

allowed_mask = df['chi2'] < (chi2_min + CUTOFF)
print(f"chi2_min = {chi2_min:.2f}  |  region permitida (chi2 < {chi2_min + CUTOFF:.2f}): "
      f"{int(allowed_mask.sum())} / {len(df)} puntos")

optuna.logging.set_verbosity(optuna.logging.WARNING)


def _train_surrogate(X, y, w, *, n_trials, timeout, base_score,
                     params_path, model_path, title):
    """Optuna + entrenamiento final de un XGBoost. Devuelve (modelo, eval_results)."""
    Xtr, Xva, ytr, yva, wtr, wva = train_test_split(
        X, y, w, test_size=0.2, random_state=42)
    dtr = xgb.DMatrix(Xtr.values, label=ytr, weight=wtr, feature_names=features)
    dva = xgb.DMatrix(Xva.values, label=yva, weight=wva, feature_names=features)
    evals = [(dtr, 'train'), (dva, 'eval')]

    def optuna_obj(trial):
        params = {
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'max_depth':        trial.suggest_int('max_depth', 2, 6),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2),
            'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'subsample':        trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 10.0),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 10.0),
            'eval_metric': 'mae', 'nthread': -1, 'base_score': base_score,
        }
        m = xgb.train(params, dtr, num_boost_round=1000, evals=evals,
                      early_stopping_rounds=20, verbose_eval=False)
        return sklearn.metrics.mean_absolute_error(yva, m.predict(dva), sample_weight=wva)

    print(f"[{title}] Optuna ({n_trials} trials, {timeout}s)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(optuna_obj, n_trials=n_trials, timeout=timeout, show_progress_bar=True)
    with open(params_path, 'w') as f:
        json.dump(study.best_params, f)

    cfg = study.best_params | {
        'objective': 'reg:squarederror', 'tree_method': 'hist',
        'eval_metric': ['mae'], 'nthread': -1, 'base_score': base_score,
    }
    eval_results = {}
    model = xgb.train(cfg, dtr, num_boost_round=2000, evals=evals,
                      early_stopping_rounds=30, verbose_eval=False,
                      evals_result=eval_results)
    model.save_model(model_path)
    print(f"[{title}] best MAE = {study.best_value:.4g}  ->  guardado en {model_path}")
    return model, eval_results


if not FORCE_RETRAIN and CLF_PATH.exists() and REG_PATH.exists():
    print(f"Modelos ya existen (CLF + REG) -- usando cache. FORCE_RETRAIN=True para reentrenar.")
else:
    # ---- (a) CLASIFICADOR: target sigmoide en TODO el espacio ----
    # Pesos: excluidos/frontera (target <= 0.8) pesan x5 para no ignorar la frontera.
    Xc = df[features]
    yc = df['target']
    wc = np.where(yc <= 0.8, 5.0, 1.0)
    model_clf, res_clf = _train_surrogate(
        Xc, yc, wc, n_trials=100, timeout=300, base_score=0.5,
        params_path=BEST_PARAMS_CLF, model_path=CLF_PATH, title='CLF')

    # ---- (b) REGRESOR: chi2 CRUDO, SOLO en la region permitida ----
    df_reg = df[allowed_mask].reset_index(drop=True)
    Xr = df_reg[features]
    yr = df_reg['chi2']
    wr = np.ones(len(df_reg))
    model_reg, res_reg = _train_surrogate(
        Xr, yr, wr, n_trials=80, timeout=240, base_score=float(yr.mean()),
        params_path=BEST_PARAMS_REG, model_path=REG_PATH, title='REG')

    # ---- Curvas de aprendizaje ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(res_clf['train']['mae'], label='train')
    axes[0].plot(res_clf['eval']['mae'],  label='val')
    axes[0].set_yscale('log'); axes[0].set_title('CLF (sigmoide) MAE')
    axes[0].set_xlabel('Iteracion'); axes[0].set_ylabel('MAE'); axes[0].legend()
    axes[0].grid(True, which='both', ls='--', alpha=0.4)
    axes[1].plot(res_reg['train']['mae'], label='train')
    axes[1].plot(res_reg['eval']['mae'],  label='val')
    axes[1].set_title(r'REG ($\chi^2$ crudo) MAE'); axes[1].set_xlabel('Iteracion')
    axes[1].set_ylabel(r'MAE en $\chi^2$'); axes[1].legend()
    axes[1].grid(True, which='both', ls='--', alpha=0.4)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "learning_curve_uv_v2.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Curvas de aprendizaje guardadas.")


## <a id="sec-4"></a>§4. Interpretabilidad SHAP

`shap.Explainer` con `model.predict` directo \u2014 dispara el **Permutation Explainer**,
que es rápido y evita los `TypeError`/`ValueError` que da el `TreeExplainer` al releer
el JSON de XGBoost. Cuatro vistas:

- **Beeswarm**: distribución de los valores SHAP por feature, color = valor del feature.
- **Bar**: importancia global media (|SHAP|).
- **Waterfall** del primer punto: cómo se construye la predicción individual.
- **Dependence (scatter)** por feature: efecto marginal y posibles interacciones.

> Nota: estos plots se muestran inline pero no se guardan automáticamente. Las versiones que están en `outputs/figures/SHAP*.png` son guardados manuales.


In [ ]:
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import numpy as np

# 1. CARGAR DATOS Y MODELO
# SHAP sobre el CLASIFICADOR (frontera permitido/excluido): nos dice que cargas
# PQ controlan si un punto entra o no en la region permitida.
df = pd.read_csv(DATASET_CSV)
features = FEATURES  # 7D incluye 'ma'
X = df[features]

model = xgb.XGBRegressor()
model.load_model(CLF_PATH)
print("Clasificador UV v2 cargado correctamente.")

# 2. PERMUTATION EXPLAINER
explainer = shap.Explainer(model.predict, X)
shap_values = explainer(X)

# 3. SUMMARY (BEE SWARM)
print("Generando Bee Swarm Plot...")
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values)

# 4. IMPORTANCIA GLOBAL (BAR)
print("Generando Bar Plot...")
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values)

# 5. WATERFALL (Punto 0)
print("Generando Waterfall Plot...")
shap.plots.waterfall(shap_values[0])

# 6. DEPENDENCE PLOTS
print("Generando Scatter Plots...")
for col in features:
    shap.plots.scatter(shap_values[:, col], color=shap_values)


## <a id="sec-5"></a>§5. Posterior MCMC + corner plot

Muestreo del posterior con **`emcee`** (`EnsembleSampler`, 128 walkers,
movimientos `DEMove`/`DESnookerMove`), como en el `04_sampling.ipynb` del tutorial.

**Posterior:**

$$
\log p(\theta) = \underbrace{-\tfrac12\,\widehat{\chi^2}_{\rm REG}(\theta)}_{\text{likelihood}}
\;+\; \underbrace{\log\pi(\theta)}_{\text{prior plano}}
\;-\; \underbrace{\mathrm{softplus}\!\big((0.5-\hat p_{\rm CLF})/0.04\big)}_{\text{muro}}
$$

- **Likelihood**: $-\tfrac12\,\chi^2$ con el **regresor** (no el sigmoide).
- **Muro**: el clasificador frena los walkers fuera de la región permitida.
- **Inicialización**: los puntos de menor $\chi^2$ del dataset (cerca del modo).
- **Convergencia**: por tiempo de autocorrelación $\tau$ (corre hasta
  $50\tau < N_{\rm pasos}$ y $\tau$ estable); luego `discard = 2\tau`, `thin = 0.5\tau`.

**Dos corner plots:**
1. **Espacio UV** — los 7 parámetros del modelo ($\log f_a$, cargas PQ, $m_a$).
2. **Observables físicos derivados** ($\log f_a$, $m_a$, $|c_V^{sb}|$, $|c_A^{\mu\mu}|$,
   $|c_G|$, $\log c\tau$, $\log\mathrm{BR}$), calculados en paralelo con `alpaca`
   sobre una **submuestra aleatoria** del posterior — es el plot comparable
   al del profesor.

Contornos a niveles $[0.393, 0.864]$ (1σ/2σ de una gaussiana 2D) y `smooth=1.5`.


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import corner
import emcee
import matplotlib.pyplot as plt
import multiprocessing as mp
import time
import warnings

# ============================================================================
# POSTERIOR  (patron de Jorge-Alda/comcha_tutorial -> 04_sampling.ipynb)
#
#   log_posterior(x) = -0.5 * REG(x)      <- likelihood real exp(-chi2/2)
#                      + log_prior(x)      <- prior (aqui: caja plana)
#                      - penalty(CLF(x))   <- muro blando: clasificador como pared
#
# OJO (correccion clave): el LIKELIHOOD viene del REGRESOR de chi2 crudo.
# El sigmoide (clasificador) NO es el likelihood: solo sirve de muro para que
# los walkers no se metan donde el regresor extrapola. Antes se muestreaba
# log(sigmoide) directamente -> meseta plana -> corner a "cajas".
# ============================================================================

clf = xgb.XGBRegressor(); clf.load_model(CLF_PATH)   # sigmoid(chi2) in [0,1]
reg = xgb.XGBRegressor(); reg.load_model(REG_PATH)   # chi2 crudo

_features = FEATURES  # 7D
_L = np.array([6.0, PQ_MIN, PQ_MIN, PQ_MIN, PQ_MIN, PQ_MIN, MA_MIN])
_H = np.array([8.0, PQ_MAX, PQ_MAX, PQ_MAX, PQ_MAX, PQ_MAX, MA_MAX])
ndim = len(_features)


def _softplus(x):
    """log(1+e^x) estable = penalizacion suave."""
    return np.logaddexp(0.0, x)


def log_posterior(theta):
    """Vectorizado: theta (n, ndim) -> log_prob (n,).  emcee con vectorize=True."""
    theta = np.atleast_2d(theta).astype(np.float32)
    out = np.full(theta.shape[0], -np.inf, dtype=float)
    in_box = np.all((theta >= _L) & (theta <= _H), axis=1)
    if in_box.any():
        t = theta[in_box]
        chi2  = reg.predict(t)              # chi2 crudo (regresor)
        p_all = clf.predict(t)              # sigmoide permitido/excluido
        log_lh  = -0.5 * chi2                                   # likelihood
        penalty = _softplus((0.5 - p_all) / 0.04)              # muro: p<0.5 -> excluido
        out[in_box] = log_lh - penalty                          # prior plano en la caja
    return out


# --- Posiciones iniciales: los puntos de menor chi2 del dataset (cerca del modo) ---
_df_seed = pd.read_csv(DATASET_CSV)
_df_seed = _df_seed[np.isfinite(_df_seed['chi2'])]
nwalkers = 128
_seed = _df_seed.nsmallest(nwalkers, 'chi2')[_features].values
_rng = np.random.default_rng(42)
p0 = _seed + 1e-3 * (_H - _L) * _rng.normal(size=_seed.shape)
p0 = np.clip(p0, _L + 1e-9, _H - 1e-9)


# --- Ejecutar / cargar MCMC ---
if not FORCE_MCMC and POSTERIOR_CSV.exists():
    print(f"Posterior ya existe -- cargando {POSTERIOR_CSV}")
    flat_samples = pd.read_csv(POSTERIOR_CSV).values
else:
    print(f"Corriendo MCMC con emcee ({nwalkers} walkers, vectorizado)...")
    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, log_posterior, vectorize=True,
        moves=[(emcee.moves.DEMove(), 0.8), (emcee.moves.DESnookerMove(), 0.2)],
    )
    max_n = 30_000
    old_tau = np.inf
    t0 = time.time()
    for _ in sampler.sample(p0, iterations=max_n, progress=True):
        if sampler.iteration % 200:
            continue
        tau = sampler.get_autocorr_time(tol=0)
        # Convergencia: cadena > 50*tau y tau estable al 1%
        converged = np.all(tau * 50 < sampler.iteration)
        converged &= np.all(np.abs(old_tau - tau) / tau < 0.01)
        old_tau = tau
        if converged:
            break

    tau = sampler.get_autocorr_time(tol=0)
    burnin = int(2 * np.max(tau))
    thinby = max(1, int(0.5 * np.min(tau)))
    flat_samples = sampler.get_chain(discard=burnin, thin=thinby, flat=True)
    acc = float(np.mean(sampler.acceptance_fraction))
    print(f"\n  pasos={sampler.iteration}  tau_max={np.max(tau):.1f}  "
          f"burn-in={burnin}  thin={thinby}  acc={acc:.3f}")
    print(f"  muestras (post burn-in + thin): {len(flat_samples):,}")

    pd.DataFrame(flat_samples, columns=_features).to_csv(POSTERIOR_CSV, index=False)
    print(f"  Guardado: {POSTERIOR_CSV}")


# --- Corner plot -- espacio UV (parametros del modelo) ---
corner_labels = [
    r"$\log_{10}(f_a)$", r"$X_{q_L}$", r"$X_{l_L}$",
    r"$X_{u_R}$", r"$X_{d_R}$", r"$X_{e_R}$", r"$m_a$ [GeV]"
]
fig1 = corner.corner(
    flat_samples, labels=corner_labels,
    show_titles=True, title_fmt=".2f",
    quantiles=[0.16, 0.5, 0.84],
    color="royalblue", smooth=1.5,
    levels=[0.393, 0.864],   # 1-sigma y 2-sigma para una gaussiana 2D
    fill_contours=True, plot_datapoints=False,
    label_kwargs={"fontsize": 11}, title_kwargs={"fontsize": 11},
)
plt.suptitle("Posterior UV v2 -- espacio de parametros del modelo", fontsize=14, y=1.01)
plt.savefig(CORNER_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Corner UV guardado en {CORNER_PNG}")


# ============================================================================
# Corner plot -- observables fisicos derivados (estilo del profesor)
#   log10(fa), ma, 100*|c_Vsb|, |c_Amm|, |cG|, log10(ctau)[m], log10 BR(B+->K+a)
# Se calculan en PARALELO sobre una submuestra ALEATORIA del posterior
# (antes se usaba flat_samples[:5000] = los primeros pasos -> nube sesgada).
# ============================================================================
from alpaca.uvmodels import PQChargedModel
from alpaca.decays.alp_decays.branching_ratios import total_decay_width
from alpaca.decays.decays import branching_ratio
from alpaca.constants import hbarc_GeVnm

try:
    _CTX = mp.get_context('fork')
except ValueError:
    _CTX = mp.get_context()


def _derive_row(row):
    logfa, pq_qL, pq_lL, pq_uR, pq_dR, pq_eR, ma = row
    fa = 10**logfa
    fa_scale = 4 * np.pi * fa
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            uv_model = PQChargedModel('non-universal model', {
                'qL': [0, 0, pq_qL], 'lL': [0, 0, pq_lL],
                'uR': pq_uR, 'dR': pq_dR, 'eR': pq_eR,
            })
            c = uv_model.get_couplings({}, fa_scale)
            clow = c.match_run(max(float(ma), 2.0), 'RL_below')
            c_Vsb = abs(complex(clow['cdL'][1, 2] + clow['cdR'][1, 2]))
            c_Amm = abs(complex(clow['ceR'][1, 1] - clow['ceL'][1, 1]))
            cG    = abs(float(clow['cG']))
            dw_dict = total_decay_width(float(ma), c, fa)
            dw_SM = float(dw_dict.get('DW_SM', 0.0))
            if dw_SM < 1e-50:
                log10_ctau = np.nan
            else:
                ctau_m = float(1e-7 * float(hbarc_GeVnm) / dw_SM) / 100.0
                log10_ctau = np.log10(ctau_m) if ctau_m > 0 else np.nan
            br = float(branching_ratio('B+ -> K+ a', float(ma), c, fa))
            log10_br = np.log10(max(br, 1e-20))
            return (logfa, ma, 100.0 * c_Vsb, c_Amm, cG, log10_ctau, log10_br)
    except Exception:
        return (logfa, ma, np.nan, np.nan, np.nan, np.nan, np.nan)


if not FORCE_MCMC and POSTERIOR_CSV_THEORY.exists():
    print(f"Posterior fisico ya existe -- cargando {POSTERIOR_CSV_THEORY}")
    theory_df = pd.read_csv(POSTERIOR_CSV_THEORY)
else:
    N_PHYS = min(15_000, len(flat_samples))
    idx = np.random.default_rng(99).choice(len(flat_samples), N_PHYS, replace=False)
    subset = flat_samples[idx]
    print(f"Calculando observables fisicos para {N_PHYS} muestras aleatorias "
          f"({N_CORES} cores)...")
    t0 = time.time()
    with _CTX.Pool(processes=N_CORES) as pool:
        rows = pool.map(_derive_row, [tuple(r) for r in subset], chunksize=8)
    theory_df = pd.DataFrame(rows, columns=[
        'log_fa', 'ma', 'c_Vsb_x100', 'c_Amm', 'cG', 'log10_ctau_m', 'log10_BR_BKa'
    ]).dropna().reset_index(drop=True)
    print(f"  {len(theory_df)}/{N_PHYS} validas en {time.time()-t0:.0f}s")
    theory_df.to_csv(POSTERIOR_CSV_THEORY, index=False)
    print(f"  Guardado: {POSTERIOR_CSV_THEORY}")

# Corner plot observables fisicos
phys_labels = [
    r"$\log_{10}(f_a/\mathrm{GeV})$",
    r"$m_a\ [\mathrm{GeV}]$",
    r"$100\times|c_V^{sb}|$",
    r"$|c_A^{\mu\mu}|$",
    r"$|c_G|$",
    r"$\log_{10}(c\tau)\ [\mathrm{m}]$",
    r"$\log_{10}\mathrm{BR}(B^+\to K^+a)$",
]
phys_data = theory_df[[
    'log_fa', 'ma', 'c_Vsb_x100', 'c_Amm', 'cG', 'log10_ctau_m', 'log10_BR_BKa'
]].values

fig2 = corner.corner(
    phys_data, labels=phys_labels,
    show_titles=True, title_fmt=".2f",
    quantiles=[0.16, 0.5, 0.84],
    color="teal", smooth=1.5,
    levels=[0.393, 0.864],
    fill_contours=True, plot_datapoints=False,
    label_kwargs={"fontsize": 10}, title_kwargs={"fontsize": 10},
)
plt.suptitle("Posterior UV v2 -- observables fisicos derivados", fontsize=14, y=1.01)
plt.savefig(CORNER_PHYS_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"Corner fisico guardado en {CORNER_PHYS_PNG}")
